In [ ]:
#| default_exp patching.tflex_inference_gen

# Inference image chunk with tflex
> Geenerating bash scripts for tflex inference

In [ ]:
#| hide
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
#| export
from typing import Any, Dict, List, Optional, Tuple, Union

In [ ]:
#| export
import sys
from pathlib import Path

In [ ]:
CV_TOOLS = Path(r'/home/ai_sintercra/homes/hasan/projects/git_data/cv_tools')
front = Path(r'/home/ai_sintercra/homes/hasan/projects/git_data/private_front_easy_pin_detection')
current_lib = Path(r'/home/ai_sintercra/homes/hasan/projects/git_data/new_test')
sys.path.append(str(CV_TOOLS))
sys.path.append(str(front))
sys.path.append(str(current_lib))


In [ ]:
#| export
from cv_tools.core import *
from cv_tools.imports import *
from cv_tools.data_processing.smb_tools import *

import json

In [ ]:
#| export
def split_images_into_chunks(
        im1_folder, # folder containing Image1 files
        im2_folder, # folder containing Image2 files
        images_per_job=50, # number of image pairs per job
        image_extensions=['.jpg', '.jpeg', '.png', '.tiff', '.bmp'], # image file extensions to include
        recursive=True)->List[List[Dict[str, str]]]: # search subfolders recursively
    """
    Split paired images into chunks for parallel processing
    
    
    """
    print("🔄 Step 1: Splitting images into chunks...")
    
    im1_path = Path(im1_folder)
    im2_path = Path(im2_folder)
    
    # Collect all im1 images
    im1_images = []
    for ext in image_extensions:
        if recursive:
            im1_images.extend(im1_path.rglob(f"*{ext}"))
            im1_images.extend(im1_path.rglob(f"*{ext.upper()}"))
        else:
            im1_images.extend(im1_path.glob(f"*{ext}"))
            im1_images.extend(im1_path.glob(f"*{ext.upper()}"))
    
    # Create paired image list
    paired_images = []
    for im1_file in im1_images:
        # Convert Image1 filename to Image2 filename
        im2_filename = im1_file.name.replace('image1', 'image2')
        im2_file = im2_path / im2_filename
        
        # Only include pairs where both files exist
        if im2_file.exists():
            paired_images.append({
                'im1': str(im1_file),
                'im2': str(im2_file)
            })
        else:
            print(f"Warning: Missing pair for {im1_file.name} -> {im2_filename}")
    
    if not paired_images:
        print(f"No paired images found between {im1_folder} and {im2_folder}")
        return []
    
    # Split into chunks
    chunks = []
    for i in range(0, len(paired_images), images_per_job):
        chunk = paired_images[i:i + images_per_job]
        chunks.append(chunk)
    
    print(f"Found {len(paired_images)} image pairs")
    print(f"Split into {len(chunks)} chunks of ~{images_per_job} pairs each")
    
    return chunks

In [ ]:
#| export
def create_bash_script_content(
    python_script_path: str, # path to the python script
    chunk_info_file: Path, # path to the chunk info file
    chunk_result_dir: Path, # path to the chunk result directory
    model_path: str, # path to the model
    python_env: str = None, # conda/virtual environment name
    additional_args: str = "", # additional arguments for the python script
    chunk_data: Dict = None # chunk data dictionary
) -> str:
    """
    Create the content for a bash script that processes one chunk
    Returns:
        Bash script content as string
    """
    
    # Environment activation
    env_activation = ""
    if python_env:
        env_activation = f"""
# Activate environment
#source activate {python_env}
# loading virtual environment, loading cuda/11.8, intializes bash as a shell
# or use: conda activate {python_env}
"""
    
    # Extract image file lists for direct passing
    im1_files = [pair['im1'] for pair in chunk_data['image_pairs']]
    im2_files = [pair['im2'] for pair in chunk_data['image_pairs']]
    
    # Create temporary file lists
    im1_list_file = chunk_result_dir / 'im1_files.txt'
    im2_list_file = chunk_result_dir / 'im2_files.txt'
    
    bash_content = f"""#!/bin/bash

#source ~/.bash_aliases

dlenv
module load cuda/11.8
#

# Tflex chunk processing script
# Generated automatically for parallel image inference
# Chunk ID: {chunk_data['chunk_id']}
# Image pairs: {len(chunk_data['image_pairs'])}
#

set -e  # Exit on any error

echo "🚀 Starting chunk {chunk_data['chunk_id']} processing..."
echo "📊 Processing {len(chunk_data['image_pairs'])} image pairs"
echo "🕐 Started at: $(date)"

{env_activation}

# Create output directory
mkdir -p {chunk_result_dir}

# Create image file lists
cat > {im1_list_file} << 'EOF'
{chr(10).join(im1_files)}
EOF

cat > {im2_list_file} << 'EOF'
{chr(10).join(im2_files)}
EOF

echo "📁 Image lists created"
echo "📄 IM1 files: {im1_list_file}"
echo "📄 IM2 files: {im2_list_file}"

# Run your Python script
echo "🐍 Running Python inference script..."
gpu_ "python {python_script_path} \\
    --model_path "{model_path}" \\
    --im1_files_path "{im1_list_file}" \\
    --im2_files_path "{im2_list_file}" \\
    --output_dir "{chunk_result_dir}" \\
    --batch_size {chunk_data['batch_size']} \\
    --num_workers {chunk_data['num_workers']} \\
    --device {chunk_data['device']} \\
    --pred_output_dir "{chunk_result_dir}/predictions" \\
    {additional_args}"

# Check if Python script succeeded
if [ $? -eq 0 ]; then
    echo "✅ Chunk {chunk_data['chunk_id']} completed successfully"
    echo "📁 Results saved to: {chunk_result_dir}"
    echo "🕐 Finished at: $(date)"
else
    echo "❌ Chunk {chunk_data['chunk_id']} failed"
    echo "🕐 Failed at: $(date)"
    exit 1
fi

echo "🎉 Chunk {chunk_data['chunk_id']} processing complete!"
"""
    
    return bash_content

In [ ]:
#| export
def generate_tflex_commands(
        im1_folder:str,
		im2_folder:str,
		output_dir:str,
		python_script_path:str,
		model_path:str,
		images_per_job:int,
		batch_size:int,
		num_workers:int,
		img_size:tuple=(1152,1632),
		device:str='cpu',
		tflex_file:str='tflex_commands.txt',
		python_env:str=None,
		additional_args:str=""

	):
	# Create output directory structure
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    
    chunks_dir = output_path / 'chunks'
    results_dir = output_path / 'results'
    logs_dir = output_path / 'logs'
    scripts_dir = output_path / 'scripts'
    
    chunks_dir.mkdir(exist_ok=True)
    results_dir.mkdir(exist_ok=True)
    logs_dir.mkdir(exist_ok=True)
    scripts_dir.mkdir(exist_ok=True)
    
    # Step 1: Split images into chunks
    chunks = split_images_into_chunks(
        im1_folder=im1_folder,
        im2_folder=im2_folder,
        images_per_job=images_per_job
    )
    
    if not chunks:
        raise ValueError("No image chunks created. Check your input folders.")
    
    # Step 2: Generate bash scripts and tflex commands
    tflex_commands = []
    
    for chunk_idx, chunk in enumerate(chunks):
        # Create chunk-specific paths
        chunk_name = f"chunk_{chunk_idx:04d}"
        chunk_result_dir = results_dir / chunk_name
        chunk_result_dir.mkdir(exist_ok=True)
        
        # Save chunk information to JSON
        chunk_info_file = chunks_dir / f"{chunk_name}_info.json"
        chunk_data = {
            'chunk_id': chunk_idx,
            'image_pairs': chunk,
            'batch_size': batch_size,
            'num_workers': num_workers,
            'img_size': img_size,
            'device': device,
            'output_dir': str(chunk_result_dir),
            'model_path': model_path
        }
        
        with open(chunk_info_file, 'w') as f:
            json.dump(chunk_data, f, indent=2)
        
        # Create bash script for this chunk
        bash_script_path = scripts_dir / f"{chunk_name}_process.sh"
        log_file = logs_dir / f"{chunk_name}.log"
        error_file = logs_dir / f"{chunk_name}.err"
        
        # Generate bash script content
        bash_content = create_bash_script_content(
            python_script_path=python_script_path,
            chunk_info_file=chunk_info_file,
            chunk_result_dir=chunk_result_dir,
            model_path=model_path,
            python_env=python_env,
            additional_args=additional_args,
            chunk_data=chunk_data
        )
        
        # Write bash script
        with open(bash_script_path, 'w') as f:
            f.write(bash_content)
        
        # Make bash script executable
        import os
        os.chmod(bash_script_path, 0o755)
        
        # Generate tflex command (just calls the bash script)
        tflex_command = f"bash {bash_script_path} > {log_file} 2> {error_file}"
        tflex_commands.append(tflex_command)
    
    # Step 3: Write tflex commands file
    tflex_file_path = output_path / tflex_file
    with open(tflex_file_path, 'w') as f:
        for cmd in tflex_commands:
            f.write(f"{cmd}\n")
    
    # Step 4: Create summary file
    summary = {
        'total_chunks': len(chunks),
        'total_image_pairs': sum(len(chunk) for chunk in chunks),
        'images_per_job': images_per_job,
        'batch_size': batch_size,
        'img_size': img_size,
        'device': device,
        'python_script': python_script_path,
        'model_path': model_path,
        'tflex_file': str(tflex_file_path),
        'output_structure': {
            'chunks_dir': str(chunks_dir),
            'results_dir': str(results_dir),
            'logs_dir': str(logs_dir),
            'scripts_dir': str(scripts_dir)
        }
    }
    
    summary_file = output_path / 'processing_summary.json'
    with open(summary_file, 'w') as f:
        json.dump(summary, f, indent=2)
    
    print(f"✅ Generated {len(tflex_commands)} bash scripts and tflex commands")
    print(f"📁 Tflex file: {tflex_file_path}")
    print(f"📊 Summary: {summary_file}")
    print(f"🔧 Total image pairs: {summary['total_image_pairs']}")
    print(f"📜 Bash scripts: {scripts_dir}")
    
    return str(tflex_file_path)




In [ ]:
#| export
@call_parse
def main_(
	im1_folder:Param("Path to first image folder", str)=r'/home/hasan/Schreibtisch/projects/data/easy_pin_detection/zero_degree_solder_pin/data/ET4/1B_solder/1B_solder_unzip/main_im1_cropped',
	im2_folder:Param("Path to second image folder", str)=r'/home/hasan/Schreibtisch/projects/data/easy_pin_detection/zero_degree_solder_pin/data/ET4/1B_solder/1B_solder_unzip/main_im2_cropped',
	output_dir:Param("Output directory for results", str)=r'/home/hasan/Schreibtisch/projects/data/easy_pin_detection/zero_degree_solder_pin/data/ET4/1B_solder/1B_solder_unzip/output',
	python_script_path:Param("Path to Python inference script", str)=r'/home/hasan/Schreibtisch/projects/git_data/new_test/nbs/51_patching.tflex_inference.ipynb',
	model_path:Param("Path to trained model", str)=r'/home/hasan/Schreibtisch/projects/data/easy_pin_detection/zero_degree_solder_pin/models/model_20250528_231243_lr_0.002_epochs_100.pth_best_val_0.9179_epoch_96.pth',
	images_per_job:Param("Number of image pairs per job", int)=50,
	batch_size:Param("Batch size", int)=1,
	num_workers:Param("Number of workers", int)=0,
	device:Param("Device", str)="cpu"

):
    print("🎯 Creating tflex commands...")
    generate_tflex_commands(
	    im1_folder=im1_folder,
	    im2_folder=im2_folder,
	    output_dir=output_dir,
	    python_script_path=python_script_path,
	    model_path=model_path,
	    images_per_job=images_per_job,
	    batch_size=batch_size,
	    num_workers=num_workers,
		device=device
    )

In [ ]:
im1_folder=r'/home/hasan/Schreibtisch/projects/data/easy_pin_detection/zero_degree_solder_pin/data/ET4/1B_solder/1B_solder_unzip/main_im1_cropped'
im2_folder=r'/home/hasan/Schreibtisch/projects/data/easy_pin_detection/zero_degree_solder_pin/data/ET4/1B_solder/1B_solder_unzip/main_im2_cropped'
output_dir=r'/home/hasan/Schreibtisch/projects/data/easy_pin_detection/zero_degree_solder_pin/data/ET4/1B_solder/1B_solder_unzip/output'
python_script_path=r'/home/hasan/Schreibtisch/projects/git_data/new_test/new_test/patching/inference_batch_image.py'
model_path=r'/home/hasan/Schreibtisch/projects/data/easy_pin_detection/zero_degree_solder_pin/models/model_20250528_231243_lr_0.002_epochs_100.pth_best_val_0.9179_epoch_96.pth'
images_per_job=50
batch_size=1
num_workers=0
main_(	
	im1_folder=im1_folder,
	im2_folder=im2_folder,
	output_dir=output_dir,
	python_script_path=python_script_path,
	model_path=model_path,
	images_per_job=images_per_job,
	batch_size=batch_size,
	num_workers=num_workers,
)

🔄 Step 1: Splitting images into chunks...
Found 3143 image pairs
Split into 63 chunks of ~50 pairs each
✅ Generated 63 bash scripts and tflex commands
📁 Tflex file: /home/hasan/Schreibtisch/projects/data/easy_pin_detection/zero_degree_solder_pin/data/ET4/1B_solder/1B_solder_unzip/output/tflex_commands.txt
📊 Summary: /home/hasan/Schreibtisch/projects/data/easy_pin_detection/zero_degree_solder_pin/data/ET4/1B_solder/1B_solder_unzip/output/processing_summary.json
🔧 Total image pairs: 3143
📜 Bash scripts: /home/hasan/Schreibtisch/projects/data/easy_pin_detection/zero_degree_solder_pin/data/ET4/1B_solder/1B_solder_unzip/output/scripts


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export('51_patching.tflex_inference.ipynb')

ValueError: '/home/hasan/Schreibtisch/projects/nbs/39_preprocessing.zero_degree_solder_pin.ipynb' is not in the subpath of '/home/hasan/Schreibtisch/projects/git_data/new_test/nbs'